# PDF Processor Notebook

Este notebook organiza o fluxo de processamento de PDF em **células pequenas e independentes** para que você possa executar etapas separadamente.

Cada seção corresponde a uma etapa do pipeline e pode ser executada isoladamente para inspeção, depuração ou ajuste.


## 1) Configurar ambiente e dependências

Esta célula importa as bibliotecas necessárias, define caminhos de entrada/saída e configura variáveis globais para as demais células.

> Execute esta célula antes de rodar as demais células dependentes.


In [1]:
import os
from pathlib import Path

import pdfplumber

# Imports locais
from utils import encontrar_pdfs, regra_divisao
from leitor_pdf import dividir_pdf

# Configurações gerais
BASE_DIR = Path(".").resolve()
PDF_DIR = BASE_DIR / "arquivos" / "pastateste"

print(f"Base: {BASE_DIR}")
print(f"Pasta de PDFs: {PDF_DIR}")


Base: C:\Users\tcunha\Documents\PDFProcessor
Pasta de PDFs: C:\Users\tcunha\Documents\PDFProcessor\arquivos\pastateste


## 2) Definir funções utilitárias

Nesta célula, definimos funções e classes reutilizáveis para extração de texto, detecção de títulos/tabelas e formatadores de saída.


In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

@dataclass
class TextoComMeta:
    texto: str
    tamanho_fonte: Optional[float] = None
    cor_fonte: Optional[List[int]] = None


def _to_rgb(color_tuple: Tuple[float, float, float]) -> List[int]:
    """Converte cores de 0-1 para 0-255."""
    return [int(round(c * 255)) for c in color_tuple]


def extrair_linhas_pagina(pagina, cortar_rodape: bool = True):
    """Extrai linhas de texto de uma página, cortando o rodapé se desejado."""
    largura = pagina.width
    altura = pagina.height - (32 if cortar_rodape else 0)

    esquerda = pagina.crop((0, 0, largura / 2, altura))
    direita = pagina.crop((largura / 2, 0, largura, altura))

    return esquerda.extract_text_lines() + direita.extract_text_lines()


def mapear_linhas_com_meta(linhas) -> List[TextoComMeta]:
    """Mapeia linhas extraídas para objetos com metadados de fonte e cor."""
    resultado = []
    for linha in linhas:
        if not linha.get("chars"):
            continue
        primeiro_char = linha["chars"][0]
        cor = primeiro_char.get("non_stroking_color")
        tamanho = primeiro_char.get("size")

        if cor and isinstance(cor, (list, tuple)) and len(cor) >= 3:
            cor_rgb = _to_rgb(tuple(cor[:3]))
        else:
            cor_rgb = None

        resultado.append(TextoComMeta(texto=linha.get("text", ""), tamanho_fonte=tamanho, cor_fonte=cor_rgb))
    return resultado


def detectar_titulos_tabelas(linhas_meta: List[TextoComMeta]) -> Dict[str, List[TextoComMeta]]:
    """Detecta linhas que parecem títulos ou tabelas com base em tamanho/cor."""
    titulos = []
    tabelas = []

    for item in linhas_meta:
        if item.tamanho_fonte and item.tamanho_fonte >= 15:
            titulos.append(item)
        if "tabela" in item.texto.lower():
            tabelas.append(item)

    return {"titulos": titulos, "tabelas": tabelas}


## 3) Carregar e pré-processar dados

Nesta seção listamos os arquivos PDF disponíveis e extraímos as linhas de texto de uma página de exemplo para pré-processamento.


In [ ]:
# Lista arquivos PDF disponíveis
pdfs = [p for p in encontrar_pdfs(str(PDF_DIR)) if p.lower().endswith(".pdf")]
print(f"Encontrados {len(pdfs)} PDFs")
for i, p in enumerate(pdfs):
    print(f"{i:02d}: {Path(p).name}")

# Escolha o índice do PDF a ser analisado
idx_pdf = 0  # <--- ajuste conforme necessário
pdf_path = pdfs[idx_pdf]
print(f"\nUsando PDF: {pdf_path}")

# Escolha a página a ser analisada
pagina_idx = 23  # <--- ajuste conforme necessário, baseado em 0-index

# Carrega e pré-processa a página
with pdfplumber.open(pdf_path) as pdf:
    pagina = pdf.pages[pagina_idx]
    linhas = extrair_linhas_pagina(pagina, cortar_rodape=True)

linhas_meta = mapear_linhas_com_meta(linhas)
print(f"Linhas extraídas: {len(linhas_meta)}")


## 4) Análise exploratória

Aqui examinamos as linhas extraídas e identificamos possíveis títulos/tabelas para entender melhor o conteúdo.


In [ ]:
# Mostra algumas linhas extraídas
for i, item in enumerate(linhas_meta[:20]):
    print(f"{i:03d} | font={item.tamanho_fonte} | color={item.cor_fonte} | text={item.texto}")

# Detecta possíveis títulos e tabelas
resultado = detectar_titulos_tabelas(linhas_meta)
print(f"\nTítulos detectados: {len(resultado['titulos'])}")
for t in resultado['titulos'][:10]:
    print(" -", t.texto)

print(f"\nTabelas detectadas: {len(resultado['tabelas'])}")
for t in resultado['tabelas'][:10]:
    print(" -", t.texto)


## 5) Treinar e avaliar modelo (processamento principal)

Aqui podemos executar a lógica principal de divisão de PDFs usando as regras configuradas em `regras_divisao` e `dividir_pdf`.


In [ ]:
# Carrega as regras de divisão (arquivo: arquivos/regras_divisao.txt)
regras = regra_divisao(str(BASE_DIR / "arquivos"))
print("Regras de divisão carregadas:", regras)

# Exemplo: aplicar a divisão ao PDF selecionado
# (A função dividir_pdf deve estar implementada em leitor_pdf.py)
try:
    saida_divisao = dividir_pdf(pdf_path, regras)
    print("Resultado da divisão:", saida_divisao)
except Exception as e:
    print("Erro ao executar dividir_pdf:", e)


## 6) Salvar resultados e artefatos

Salve saídas intermediárias (por exemplo, linhas extraídas ou resultados de divisão) em arquivos para evitar reprocessar o mesmo PDF repetidamente.


In [ ]:
import json

saida_dir = BASE_DIR / "saida_notebook"
saida_dir.mkdir(exist_ok=True)

# Salva linhas meta em JSON
saida_json = saida_dir / f"linhas_meta_{Path(pdf_path).stem}.json"
with open(saida_json, "w", encoding="utf-8") as f:
    json.dump([item.__dict__ for item in linhas_meta], f, ensure_ascii=False, indent=2)

print("Salvo:", saida_json)

# Se houver resultado da divisão, salve também
if 'saida_divisao' in globals():
    saida_divisao_path = saida_dir / f"saida_divisao_{Path(pdf_path).stem}.json"
    with open(saida_divisao_path, "w", encoding="utf-8") as f:
        json.dump(saida_divisao, f, ensure_ascii=False, indent=2)
    print("Salvo divisão:", saida_divisao_path)
